In [ ]:
import duckdb
import pandas as pd

from IPython.display import display
pd.options.display.max_columns = None
pd.options.display.max_rows = 30
pd.get_option("display.max_rows")

import os
from pathlib import Path

con = duckdb.connect()
con.sql("""SHOW TABLES""").show()

raw_data_path = Path("/Volumes/SANDISKUSBC/MLOps/hdb_price_prediction_mlops/data/raw")
os.listdir(raw_data_path)

In [ ]:
train_file_path = raw_data_path/'train.csv'
con.execute(f"CREATE TABLE train AS SELECT * FROM '{train_file_path}'")


eval_file_path = raw_data_path/'eval.csv'
con.execute(f"CREATE TABLE eval AS SELECT * FROM '{eval_file_path}'")

con.sql("""SHOW TABLES""").show()

In [ ]:
#con.sql("""describe  train""").show()
#con.sql("""describe  eval""").show()


In [ ]:
storey_range_ls = con.sql("""SELECT DISTINCT storey_range FROM train ORDER BY  storey_range""").df()['storey_range'].tolist()
print(storey_range_ls)
con.sql("""SELECT * FROM train""").df()

In [ ]:
con.sql("""SELECT * FROM eval""").df()

### CHECK DUPLICATES:

In [ ]:
def findDups(tablename):
    qry = f"""
        SELECT 
        month
        ,blk_no
        ,road_name
        ,building
        ,postal
        ,storey_range
        ,resale_price
        ,flat_type
        ,flat_model
        ,floor_area_sqm
        ,distance_to_pri_school_meters
        ,latitude
        ,longitude
        ,COUNT(*) AS Occurrence
        FROM {tablename}
        GROUP BY 
        month
        ,blk_no
        ,road_name
        ,building
        ,postal
        ,storey_range
        ,resale_price
        ,flat_type
        ,flat_model
        ,floor_area_sqm
        ,distance_to_pri_school_meters
        ,latitude
        ,longitude
        HAVING COUNT(*) > 1
        """
    return con.sql(qry).df()

In [ ]:
findDups('train')

In [ ]:
findDups('eval')

In [ ]:
def dropDupSQL(tbl,newtbl):
    qry = f"""
    CREATE TABLE {newtbl} AS
    SELECT DISTINCT *
    FROM {tbl};
    DROP TABLE {tbl};
    """
    con.sql(qry)
    return con.sql(f"""SELECT * FROM {newtbl}""").df()

In [ ]:
dropDupSQL('train','dedup_train')

In [ ]:
dropDupSQL('eval','dedup_eval')

In [ ]:
con.sql("""SHOW TABLES""").show()

In [ ]:
storey_range_ls

### Boxplots:

In [ ]:
import plotly.express as px

# Fetch data from DuckDB table
df_plot = con.sql("SELECT flat_model,storey_range,town, flat_type,region_ura, resale_price FROM dedup_train").df()

# 1. Interactive Boxplot: Resale Price by Town
# We sort by town name alphabetically for easy navigation
fig_town = px.box(
    df_plot, 
    x="town", 
    y="resale_price", 
    title="Interactive Resale Price Distribution by Town",
    category_orders={"town": sorted(df_plot['town'].unique())},
    labels={"town": "Town", "resale_price": "Resale Price ($)"},
    template="plotly_dark",
    color = "town",
    height=600
)
fig_town.update_xaxes(tickangle=45)
fig_town.show()


# 2. Interactive Boxplot: Resale Price by region_ura
# We sort by town name alphabetically for easy navigation
fig_region_ura = px.box(
    df_plot, 
    x="region_ura", 
    y="resale_price", 
    title="Interactive Resale Price Distribution by region_ura",
    category_orders={"region_ura": sorted(df_plot['region_ura'].unique())},
    labels={"region_ura": "region_ura", "resale_price": "Resale Price ($)"},
    template="plotly_dark",
    color= "region_ura",
    height=600
)
fig_region_ura.update_xaxes(tickangle=45)
fig_region_ura.show()

# 3. Interactive Boxplot: Resale Price by Flat Type
# We order by logical size of the flats
flat_order = ['1 ROOM', '2 ROOM', '3 ROOM', '4 ROOM', '5 ROOM', 'EXECUTIVE', 'MULTI-GENERATION']
fig_flat = px.box(
    df_plot, 
    x="flat_type", 
    y="resale_price", 
    title="Interactive Resale Price Distribution by Flat Type",
    category_orders={"flat_type": [f for f in flat_order if f in df_plot['flat_type'].unique()]},
    labels={"flat_type": "Flat Type", "resale_price": "Resale Price ($)"},
    template="plotly_dark",
    color="flat_type", # Color-coding makes it more readable
    height=600
)
fig_flat.show()


# 4. Interactive Boxplot: Resale Price by storey_range
# We order by logical size of the flats
storey_order = storey_range_ls
fig_storey = px.box(
    df_plot, 
    x="storey_range", 
    y="resale_price", 
    title="Interactive Resale Price Distribution by Storey Range",
    category_orders={"storey_range": [f for f in storey_order if f in df_plot['storey_range'].unique()]},
    labels={"storey_range": "Storey Range", "resale_price": "Resale Price ($)"},
    template="plotly_dark",
    color="storey_range", # Color-coding makes it more readable
    height=600
)
fig_storey.show()


# 5. Interactive Boxplot: Resale Price by flat_model
# We sort by town name alphabetically for easy navigation
fig_flat_model = px.box(
    df_plot, 
    x="flat_model", 
    y="resale_price", 
    title="Interactive Resale Price Distribution by flat_model",
    category_orders={"flat_model": sorted(df_plot['flat_model'].unique())},
    labels={"flat_model": "flat_model", "resale_price": "Resale Price ($)"},
    template="plotly_dark",
    color= "flat_model",
    height=600
)
fig_flat_model.update_xaxes(tickangle=45)
fig_flat_model.show()

In [ ]:
### SAVE CLEAN DF
clean_train_df = con.sql("""SELECT * FROM dedup_train""").df()
clean_eval_df = con.sql("""SELECT * FROM dedup_eval""").df()

clean_train_df.to_csv(raw_data_path/'clean_train.csv',index=None)
clean_eval_df.to_csv(raw_data_path/'clean_eval.csv',index=None)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 1. Filter only numerical columns
numerical_df = clean_train_df.select_dtypes(include=['number'])

# 2. Calculate the correlation matrix
corr_matrix = numerical_df.corr()

# 3. Create a professional, high-end heatmap
plt.figure(figsize=(16, 12))

# Mask the upper triangle (since correlation matrices are symmetrical)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# Use a premium diverging color palette
cmap = sns.diverging_palette(230, 20, as_cmap=True)

# Plot the heatmap
sns.heatmap(
    corr_matrix, 
    mask=mask, 
    cmap=cmap, 
    vmax=1.0, 
    center=0,
    annot=True,        # Show correlation values
    fmt=".2f",         # Format to 2 decimal places
    square=True,       # Keep boxes square
    linewidths=.5,     # Add subtle borders between cells
    cbar_kws={"shrink": .7}
)

plt.title("Correlation Matrix of Numerical Features", fontsize=22, fontweight='bold', pad=25)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
